# Werewolf Transformer — pilot-002 fixed-policy evaluation

このノートブックは **学習を一切しません**。pilot-002 で完成した直近の policy を固定し、同じ条件で再評価します。

各陣営の直近3 policyを使って 3×3×3 = 27 profile を作り、各 profile を20村ずつ評価します（合計540村）。
評価テーブルは Google Drive に逐次保存されるため、Colab が切れた場合も同じノートブックを上から再実行すれば不足分だけ補います。

pilot-002完了時のpoolなら、評価対象は通常:
- Village: g000078 / g000079 / g000082
- Werewolf: g000078 / g000080 / g000083
- Fox: g000078 / g000081 / g000084

最後に表示される `FIXED EVALUATION REPORT` を ChatGPT に貼ってください。


In [ ]:
# 1) Google Drive を接続
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) 最新 main を取得して依存関係をインストール
%cd /content
!rm -rf Are-you-werewolf
!git clone --depth 1 https://github.com/dolphin23-jp/Are-you-werewolf.git
%cd /content/Are-you-werewolf/backend
!python -m pip install -q -e ".[rl,transformer]"


In [ ]:
# 3) GPU・pilot-002 pool・評価対象を確認
from pathlib import Path
import torch
from app.engine.roles import Team
from app.training.torch_pool import TorchPolicyPool

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU が有効ではありません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでください。'
    )

RUN_ROOT = Path('/content/drive/MyDrive/werewolf-training/pilot-002')
POOL = RUN_ROOT / 'pool'
EVAL = RUN_ROOT / 'fixed-evaluation-last3-v1'
EVAL.mkdir(parents=True, exist_ok=True)

if not (POOL / 'manifest.json').exists():
    raise RuntimeError('pilot-002 pool が見つかりません。pilot-002 が完了した Google Drive を接続してください。')

pool = TorchPolicyPool(POOL, device='cpu')
targets = {
    team.value: pool.policy_ids_for_team(team, last=3)
    for team in Team
}
if any(len(ids) != 3 for ids in targets.values()):
    raise RuntimeError(f'各陣営3 policyを取得できません: {targets}')

print('GPU:', torch.cuda.get_device_name(0))
print('pool generations:', pool.next_generation)
print('evaluation targets:', targets)
print('evaluation dir:', EVAL)


In [ ]:
# 4) 27 profile × 20村 = 540村を固定評価
#    payoffs.json が途中まで存在する場合、不足分だけ追加します。学習は行いません。
import subprocess

table = EVAL / 'payoffs.json'
measure_log = EVAL / 'measure.log'
cmd = [
    'python', 'scripts/measure_population_payoffs_torch.py',
    '--pool-dir', str(POOL),
    '--table', str(table),
    '--last', '3',
    '--games-per-profile', '20',
    '--extra-games', '0',
    '--seed', '3101',
    '--parallel-games', '16',
    '--inference-batch-size', '64',
    '--device', 'auto',
]

with measure_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print('\nFIXED PAYOFF MEASUREMENT COMPLETE')


In [ ]:
# 5) 評価結果だけから empirical meta strategy を解く
meta = EVAL / 'meta.json'
solve_log = EVAL / 'solve.log'
cmd = [
    'python', 'scripts/solve_population_meta_torch.py',
    '--table', str(table),
    '--pool-dir', str(POOL),
    '--output', str(meta),
    '--last', '3',
    '--temperature', '0.25',
    '--iterations', '100',
    '--damping', '0.5',
    '--device', 'cpu',
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
solve_log.write_text(result.stdout, encoding='utf-8')
print(result.stdout)


In [ ]:
# 6) ChatGPT に送るレポートを表示
import json

print('\n===== FIXED EVALUATION REPORT =====')
print('targets:', json.dumps(targets, ensure_ascii=False))
print('\n===== META SOLVER =====')
print(solve_log.read_text(encoding='utf-8').strip())
print('===== END META SOLVER =====')

print('\n===== PROFILE PAYOFFS =====')
for line in measure_log.read_text(encoding='utf-8').splitlines():
    if line.startswith('profile=') or line.startswith('measured_profiles='):
        print(line)
print('===== END PROFILE PAYOFFS =====')
print('===== END FIXED EVALUATION REPORT =====')
print('\nこの FIXED EVALUATION REPORT 全体を ChatGPT に貼り付けてください。')


## Colab が途中で切れた場合

同じノートブックを上から再実行してください。`payoffs.json` に既に記録済みのprofileは再利用し、20村に足りないprofileだけ追加評価します。
この評価ではモデル重みを更新しないため、pilot-002 の policy pool は変更されません。
